# 00 — Environment test

**Environment checks only.** Run this first from a fresh kernel after creating the pinned virtual environment. It confirms that this clone can import the required data-science and GIS stack, resolves the repository root portably, and performs only in-memory smoke tests. It does not download data, create derived project outputs, or run the analytical pipeline.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_support import resolve_project_root
PROJECT_ROOT = resolve_project_root(PROJECT_ROOT)
print('Repository root:', PROJECT_ROOT)
print('Python executable:', sys.executable)
print('Python version:', sys.version.split()[0])

Repository root: C:\Personal\private\Portugal\Study\DataScience\Project\Reproducible_Wildfire_Exposure_Capstone
Python executable: c:\Personal\private\Portugal\Study\DataScience\Project\Reproducible_Wildfire_Exposure_Capstone\.venv\Scripts\python.exe
Python version: 3.13.11


## Local-input preflight

This calls the same read-only source-presence check as `scripts/run_project.py --mode preflight`. It does not open, download, alter, or validate the contents of raw files.

In [ ]:
import pandas as pd
from src.project_run import raw_data_preflight

preflight = raw_data_preflight()
display(pd.DataFrame(preflight['groups']))
print('Raw-input preflight status:', preflight['status'])

,source_group,expected_files,present_files,missing_files,status
0,ICNF annual burned areas,18,18,[],ready
1,CAOP 2025,1,1,[],ready
2,Copernicus CLC packages,3,3,[],ready
3,Copernicus DEM GLO-30 tiles,21,21,[],ready
4,ERA5-Land JJAS GRIBs,18,18,[],ready
5,ICNF structural-hazard raster,1,1,[],ready


Raw-input preflight status: ready


## Pinned package contract

The table compares the installed package versions with the exact pins in `requirements.txt`. A mismatch should be corrected by rebuilding the local virtual environment before running the reproducible pipeline.

In [ ]:
import importlib.metadata as metadata
from src.notebook_support import pinned_requirements

required = pinned_requirements(PROJECT_ROOT)
package_versions = pd.DataFrame([
    {'package': package, 'required': expected, 'installed': metadata.version(package), 'matches': metadata.version(package) == expected}
    for package, expected in required.items()
])
assert package_versions['matches'].all(), 'At least one installed package differs from requirements.txt.'
display(package_versions)

,package,required,installed,matches
0,numpy,2.3.5,2.3.5,True
1,pandas,2.2.3,2.2.3,True
2,matplotlib,3.10.8,3.10.8,True
3,scikit-learn,1.8.0,1.8.0,True
4,geopandas,1.1.2,1.1.2,True
5,shapely,2.1.2,2.1.2,True
6,pyproj,3.7.2,3.7.2,True
7,pyogrio,0.12.1,0.12.1,True
8,rasterio,1.5.0,1.5.0,True
9,requests,2.32.5,2.32.5,True


## Geospatial and machine-learning smoke tests

These small synthetic examples confirm CRS transformation and numerical preprocessing only. They do not use project data or write figures.

In [ ]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from sklearn.preprocessing import StandardScaler
from src.config import SPATIAL

sample = gpd.GeoDataFrame(
    {'location': ['Lisbon test point', 'Porto test point']},
    geometry=[Point(-9.1393, 38.7223), Point(-8.6291, 41.1579)],
    crs='EPSG:4326',
)
projected = sample.to_crs(SPATIAL.analysis_crs)
assert projected.crs.to_string() == SPATIAL.analysis_crs == 'EPSG:3763'

synthetic_features = pd.DataFrame({'forest_shrub_share_2km': [0.10, 0.25, 0.70, 0.85], 'fire_years_previous_10y_2km': [0, 1, 3, 5]})
scaled = StandardScaler().fit_transform(synthetic_features)
assert np.isfinite(scaled).all()
display(projected)
print('EPSG:3763 transformation and in-memory numerical-preprocessing checks passed.')

,location,geometry
0,Lisbon test point,POINT (-87503.439 -104538.892)
1,Porto test point,POINT (-41630.673 165532.264)


EPSG:3763 transformation and in-memory numerical-preprocessing checks passed.


## Next step

Run `python scripts/run_project.py --mode preflight` in a terminal to check that local raw inputs are present without downloading or changing them. Then continue with `01_data_collection.ipynb`.